# MediCore Monitoring Data Analysis

This notebook analyses monitoring data collected from the MediCore cloud infrastructure deployment.

The aim is to evaluate system performance, security events and storage growth trends. The results will be used to support operational decision making, risk management and compliance reporting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

df = pd.read_csv('medicore_monitoring_data.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(df.shape)
df.head()

# Figure 1 – Average CPU Usage Over Time

This visualisation evaluates average CPU utilisation across the MediCore cloud infrastructure. A 7-day moving average has been included to smooth short-term fluctuations and identify long-term performance trends.

The highest recorded CPU utilisation was 84.3%, indicating periods of elevated system demand. The moving average helps distinguish long-term workload patterns from short-term spikes.

The results support the use of Auto Scaling controls and demonstrate that the deployed AWS environment is capable of supporting MediCore's workload of approximately 140,000 patient records while maintaining adequate performance.

In [ ]:
cpu_df = (
    df[df['vm_id'].str.contains('medicore', case=False)]
    .groupby('timestamp')['cpu_usage_pct']
    .mean()
    .reset_index()
)

cpu_df['cpu_7d_ma'] = (
    cpu_df['cpu_usage_pct']
    .rolling(window=168, min_periods=1)
    .mean()
)

peak_row = cpu_df.loc[cpu_df['cpu_usage_pct'].idxmax()]

plt.figure(figsize=(14,6))
plt.plot(cpu_df['timestamp'], cpu_df['cpu_usage_pct'], label='Average CPU Usage (%)')
plt.plot(cpu_df['timestamp'], cpu_df['cpu_7d_ma'], linewidth=3, label='7-Day Moving Average')
plt.scatter(peak_row['timestamp'], peak_row['cpu_usage_pct'], color='red')

plt.annotate(
    f"Peak = {peak_row['cpu_usage_pct']:.1f}%",
    (peak_row['timestamp'], peak_row['cpu_usage_pct'])
)

plt.title('Figure 1 - Average CPU Usage Over Time')
plt.xlabel('Date')
plt.ylabel('CPU Usage (%)')
plt.legend()
plt.tight_layout()
plt.savefig('figure1-cpu-usage.png')
plt.show()

print('Peak CPU Usage:', round(peak_row['cpu_usage_pct'],2))

## Figure 1 Analysis

Figure 1 presents average CPU utilisation across the MediCore cloud infrastructure. The highest recorded CPU utilisation was 84.3%, indicating periods of increased demand. The 7-day moving average smooths short-term fluctuations and provides a clearer picture of overall workload trends.

The results suggest that workload levels increase during operational periods and support the decision to deploy Auto Scaling controls. Although utilisation exceeded the 70% scaling threshold at several points, the environment remained capable of supporting operations. Continued monitoring should be maintained to identify future capacity constraints and support proactive infrastructure planning.

# Figure 2 – Failed SSH Login Attempts by Hour of Day

This visualisation analyses failed SSH login activity across the MediCore cloud infrastructure to identify periods of increased authentication failures and potential brute-force attacks.

In [ ]:
login_df = (
    df.groupby('hour')['failed_ssh_logins']
    .sum()
    .reset_index()
)

peak_row = login_df.loc[login_df['failed_ssh_logins'].idxmax()]

plt.figure(figsize=(10,5))
sns.barplot(data=login_df, x='hour', y='failed_ssh_logins', color='steelblue')

plt.axvline(
    x=peak_row['hour'],
    color='red',
    linestyle='--'
)

plt.title('Figure 2 - Failed SSH Login Attempts by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Failed SSH Logins')
plt.tight_layout()
plt.savefig('figure2-failed-logins.png')
plt.show()

print('Peak failed login hour:', int(peak_row['hour']))
print('Failed logins:', int(peak_row['failed_ssh_logins']))

## Figure 2 Analysis

Figure 2 shows failed SSH login activity grouped by hour of day. Authentication failures are an indicator of attempted unauthorised access and potential brute-force attacks against cloud infrastructure.

The highest concentration of failed login attempts occurred at 15:00, where 1,949 failed SSH login attempts were recorded. This period represents the most significant authentication threat observed within the monitoring dataset.

These findings support the continued use of bastion-host restrictions, CloudWatch monitoring alerts and least-privilege access controls. Additional controls such as automated lockout policies and intrusion-prevention mechanisms should also be considered to reduce the likelihood of successful brute-force attacks.

# Figure 3 – Storage Usage Trend and Capacity Forecast

This visualisation analyses storage growth across the MediCore infrastructure to identify long-term capacity trends and support infrastructure planning.

In [ ]:
storage_df = (
    df[df['vm_id'].str.contains('medicore', case=False)]
    .groupby('timestamp')['storage_used_gb']
    .mean()
    .reset_index()
)

storage_df['hours'] = range(len(storage_df))
coef = np.polyfit(storage_df['hours'], storage_df['storage_used_gb'], 1)
trend = np.poly1d(coef)
storage_df['forecast'] = trend(storage_df['hours'])

plt.figure(figsize=(12,6))
plt.plot(storage_df['timestamp'], storage_df['storage_used_gb'], label='Actual Storage Usage')
plt.plot(storage_df['timestamp'], storage_df['forecast'], linestyle='--', label='Trend Line')
plt.title('Figure 3 - Storage Usage Trend and Capacity Forecast')
plt.xlabel('Date')
plt.ylabel('Storage Used (GB)')
plt.legend()
plt.tight_layout()
plt.savefig('figure3-storage-trend.png')
plt.show()

## Figure 3 Analysis

Figure 3 shows a consistent increase in storage utilisation throughout the monitoring period. Storage consumption increased from approximately 65 GB to more than 105 GB, demonstrating sustained data growth across the MediCore environment.

This trend is expected within healthcare organisations because of increasing patient records, monitoring data, audit logs and backup requirements. Continued storage growth will eventually require additional capacity planning. Proactive monitoring allows infrastructure teams to forecast future demand and allocate resources before service availability is affected.

# Executive Summary

The MediCore monitoring data demonstrates predictable workload increases, identifiable security threats and ongoing storage growth across the infrastructure environment.

CPU utilisation analysis identified a peak value of 84.3%, validating the deployment of Auto Scaling controls to support changing demand levels. Security analysis identified a significant concentration of failed SSH login attempts at 15:00, where 1,949 failures were recorded, reinforcing the need for continued monitoring and access-control restrictions. Storage analysis demonstrated consistent growth throughout the reporting period and highlighted the importance of future capacity planning.

Overall, the cloud deployment provides an effective balance of security, scalability and operational performance while maintaining sufficient monitoring capabilities to support future growth and regulatory compliance.